In [ ]:
# BLOCO 1 - PREPARACAO UNICA DA BASE E SPLIT TREINO/TESTE
from pathlib import Path
import pickle
import unicodedata

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, roc_auc_score
import lightgbm as lgb

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def remover_acentos(texto):
    texto = "" if pd.isna(texto) else str(texto)
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def limpar_colunas(df):
    df = df.copy()
    df.columns = (
        df.columns
        .map(remover_acentos)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
    )
    return df


def limpar_moeda(col):
    return (
        col.astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )


def to_numeric_safe(col):
    return pd.to_numeric(col, errors="coerce")


def carregar_base_modelagem():
    candidatos = pd.read_csv(
        "Hackaton_Enter_Base_Candidatos.xlsx - Subsídios disponibilizados.csv",
        header=1,
    )
    resultados = pd.read_csv(
        "Hackaton_Enter_Base_Candidatos.xlsx - Resultados dos processos.csv"
    )

    candidatos = limpar_colunas(candidatos)
    resultados = limpar_colunas(resultados)

    for df_temp in [candidatos, resultados]:
        for col in df_temp.columns:
            if "processo" in col:
                df_temp.rename(columns={col: "numero_do_processo"}, inplace=True)
                break

    candidatos["numero_do_processo"] = candidatos["numero_do_processo"].astype(str)
    resultados["numero_do_processo"] = resultados["numero_do_processo"].astype(str)

    df_base = candidatos.merge(resultados, on="numero_do_processo", how="left")

    valor_causa_col = next((col for col in df_base.columns if "valor" in col and "causa" in col), None)
    valor_condenacao_col = next(
        (
            col for col in df_base.columns
            if "valor" in col and "conden" in col and "inden" in col
        ),
        None,
    )

    if valor_causa_col is None or valor_condenacao_col is None:
        raise KeyError("Nao foi possivel localizar as colunas de valores monetarios.")

    df_base["valor_da_causa"] = to_numeric_safe(limpar_moeda(df_base[valor_causa_col]))
    df_base["valor_condenacao"] = to_numeric_safe(limpar_moeda(df_base[valor_condenacao_col]))

    doc_cols = [
        "contrato",
        "extrato",
        "comprovante_de_credito",
        "dossie",
        "demonstrativo_de_evolucao_da_divida",
        "laudo_referenciado",
    ]

    for col in doc_cols:
        df_base[col] = to_numeric_safe(df_base[col]).fillna(0).astype(int)

    df_base["qtd_docs"] = df_base[doc_cols].sum(axis=1)
    df_base["docs_faltantes"] = len(doc_cols) - df_base["qtd_docs"]
    df_base["doc_ratio"] = df_base["qtd_docs"] / len(doc_cols)
    df_base["doc_score"] = (
        3 * df_base["contrato"]
        + 3 * df_base["extrato"]
        + 2 * df_base["comprovante_de_credito"]
        + 1 * df_base["demonstrativo_de_evolucao_da_divida"]
    )
    df_base["tem_todos_docs"] = (df_base["qtd_docs"] == len(doc_cols)).astype(int)
    df_base["docs_essenciais"] = df_base[["contrato", "extrato", "comprovante_de_credito"]].sum(axis=1)
    df_base["docs_probatorios"] = df_base[["dossie", "demonstrativo_de_evolucao_da_divida", "laudo_referenciado"]].sum(axis=1)
    df_base["combo_docs"] = (
        df_base["contrato"].astype(str) + "_" +
        df_base["extrato"].astype(str) + "_" +
        df_base["comprovante_de_credito"].astype(str)
    )

    for col in ["assunto", "sub_assunto", "uf"]:
        df_base[col] = df_base[col].fillna("desconhecido").astype(str)

    df_base["assunto_sub_assunto"] = df_base["assunto"] + "__" + df_base["sub_assunto"]
    df_base["uf_assunto"] = df_base["uf"] + "__" + df_base["assunto"]

    df_base["perdeu"] = df_base["resultado_micro"].isin(["Procedência", "Parcial procedência"]).astype(int)

    return df_base, doc_cols


DF_FULL, DOC_COLS = carregar_base_modelagem()
TRAIN_IDX, TEST_IDX = train_test_split(
    DF_FULL.index,
    test_size=0.2,
    random_state=42,
    stratify=DF_FULL["perdeu"],
)

DF_TRAIN = DF_FULL.loc[TRAIN_IDX].copy()
DF_TEST = DF_FULL.loc[TEST_IDX].copy()

np.save(ARTIFACT_DIR / "train_idx.npy", np.array(TRAIN_IDX))
np.save(ARTIFACT_DIR / "test_idx.npy", np.array(TEST_IDX))

print("Base pronta.")
print("Treino:", DF_TRAIN.shape, "| Teste:", DF_TEST.shape)
print("Checkpoints em:", ARTIFACT_DIR.resolve())

Base pronta.
Treino: (48000, 26) | Teste: (12000, 26)
Checkpoints em: D:\João Vitor\Faculdade graduação SI\HackatonEnter\repos\analise_exploratoria\pythonProject\artifacts


In [ ]:
# BLOCO 2 - TREINO E CHECKPOINT DO MODELO DE RISCO (P(perder))
RISK_FEATURES = DOC_COLS + [
    "qtd_docs",
    "docs_faltantes",
    "doc_ratio",
    "doc_score",
    "tem_todos_docs",
    "docs_essenciais",
    "docs_probatorios",
    "combo_docs",
    "assunto",
    "sub_assunto",
    "uf",
    "assunto_sub_assunto",
    "uf_assunto",
]


def make_risk_matrix(df_input, expected_cols=None):
    x = pd.get_dummies(df_input[RISK_FEATURES].copy(), drop_first=True)
    if expected_cols is not None:
        x = x.reindex(columns=expected_cols, fill_value=0)
    return x


X_risk_train = make_risk_matrix(DF_TRAIN)
y_risk_train = DF_TRAIN["perdeu"]

risk_weights = y_risk_train.map({0: 1.0, 1: 2.0})

RISK_MODEL = lgb.LGBMClassifier(
    n_estimators=700,
    learning_rate=0.03,
    num_leaves=63,
    random_state=42,
)
RISK_MODEL.fit(X_risk_train, y_risk_train, sample_weight=risk_weights)

X_risk_test = make_risk_matrix(DF_TEST, expected_cols=X_risk_train.columns)
y_risk_test = DF_TEST["perdeu"]
P_PERDER_TEST = RISK_MODEL.predict_proba(X_risk_test)[:, 1]
y_risk_pred = (P_PERDER_TEST >= 0.5).astype(int)

print("===== MODELO DE RISCO (TESTE NAO VISTO) =====")
print("AUC:", roc_auc_score(y_risk_test, P_PERDER_TEST))
print(classification_report(y_risk_test, y_risk_pred, target_names=["Ganhou", "Perdeu"]))

risk_checkpoint = {
    "model": RISK_MODEL,
    "risk_features": RISK_FEATURES,
    "feature_columns": list(X_risk_train.columns),
}

with open(ARTIFACT_DIR / "risk_model.pkl", "wb") as f:
    pickle.dump(risk_checkpoint, f)

print("Checkpoint risco salvo em:", ARTIFACT_DIR / "risk_model.pkl")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 14390, number of negative: 33610
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008680 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 171
[LightGBM] [Info] Number of data points in the train set: 48000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.461292 -> initscore=-0.155143
[LightGBM] [Info] Start training from score -0.155143
===== MODELO DE RISCO (TESTE NAO VISTO) =====
AUC: 0.9179548052509544
              precision    recall  f1-score   support

      Ganhou       0.92      0.87      0.90      8403
      Perdeu       0.74      0.83      0.78      3597

    accuracy                           0.86     12000
   macro avg       0.83      0.85      0.84     12000
weighted avg       0.

In [ ]:
# BLOCO 3 - TREINO E CHECKPOINT DO MODELO DE CUSTO (valor_condenacao_estimado)
COST_FEATURES = [
    "valor_da_causa",
    "qtd_docs",
    "docs_faltantes",
    "doc_ratio",
    "doc_score",
    "tem_todos_docs",
    "docs_essenciais",
    "docs_probatorios",
    "assunto",
    "sub_assunto",
    "uf",
    "assunto_sub_assunto",
    "uf_assunto",
    "combo_docs",
] + DOC_COLS


def make_cost_matrix(df_input, expected_cols=None):
    x = pd.get_dummies(df_input[COST_FEATURES].copy(), drop_first=True)
    if expected_cols is not None:
        x = x.reindex(columns=expected_cols, fill_value=0)
    return x


# Treino do custo apenas em casos de perda com valor observado > 0.
DF_TRAIN_COST = DF_TRAIN[
    (DF_TRAIN["perdeu"] == 1)
    & DF_TRAIN["valor_condenacao"].notna()
    & (DF_TRAIN["valor_condenacao"] > 0)
].copy()

DF_TEST_COST = DF_TEST[
    (DF_TEST["perdeu"] == 1)
    & DF_TEST["valor_condenacao"].notna()
    & (DF_TEST["valor_condenacao"] > 0)
].copy()

X_cost_train = make_cost_matrix(DF_TRAIN_COST)
y_cost_train = np.log1p(DF_TRAIN_COST["valor_condenacao"])

COST_MODEL = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=1200,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
)
COST_MODEL.fit(X_cost_train, y_cost_train)

X_cost_test = make_cost_matrix(DF_TEST_COST, expected_cols=X_cost_train.columns)
y_cost_test_real = DF_TEST_COST["valor_condenacao"]
y_cost_test_pred = np.expm1(COST_MODEL.predict(X_cost_test))

baseline_cost = np.repeat(np.expm1(y_cost_train.mean()), len(y_cost_test_real))

print("===== MODELO DE CUSTO (TESTE NAO VISTO) =====")
print("Amostras treino custo:", len(DF_TRAIN_COST))
print("Amostras teste custo:", len(DF_TEST_COST))
print("MAE:", mean_absolute_error(y_cost_test_real, y_cost_test_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_cost_test_real, y_cost_test_pred)))
print("R2:", r2_score(y_cost_test_real, y_cost_test_pred))
print("MAE baseline:", mean_absolute_error(y_cost_test_real, baseline_cost))

cost_checkpoint = {
    "model": COST_MODEL,
    "cost_features": COST_FEATURES,
    "feature_columns": list(X_cost_train.columns),
}

with open(ARTIFACT_DIR / "cost_model.pkl", "wb") as f:
    pickle.dump(cost_checkpoint, f)

print("Checkpoint custo salvo em:", ARTIFACT_DIR / "cost_model.pkl")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001028 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 426
[LightGBM] [Info] Number of data points in the train set: 14390, number of used features: 73
[LightGBM] [Info] Start training from score 9.225052
===== MODELO DE CUSTO (TESTE NAO VISTO) =====
Amostras treino custo: 14390
Amostras teste custo: 3597
MAE: 2320.5744158601733
RMSE: 2926.0223813815305
R2: 0.5938273247203664
MAE baseline: 3761.0305161216697
Checkpoint custo salvo em: artifacts\cost_model.pkl


In [ ]:
# BLOCO 4 - FUNCOES DE INFERENCIA + COMBINACAO E ECONOMIA (expected_loss)

with open(ARTIFACT_DIR / "risk_model.pkl", "rb") as f:
    RISK_CKPT = pickle.load(f)

with open(ARTIFACT_DIR / "cost_model.pkl", "rb") as f:
    COST_CKPT = pickle.load(f)


def score_expected_loss(df_input, risk_ckpt, cost_ckpt):
    x_risk = make_risk_matrix(df_input, expected_cols=risk_ckpt["feature_columns"])
    p_perder = risk_ckpt["model"].predict_proba(x_risk)[:, 1]

    x_cost = make_cost_matrix(df_input, expected_cols=cost_ckpt["feature_columns"])
    valor_estimado = np.expm1(cost_ckpt["model"].predict(x_cost))
    valor_estimado = np.clip(valor_estimado, 0, np.quantile(valor_estimado, 0.995))

    out = df_input.copy()
    out["prob_perder"] = p_perder
    out["valor_condenacao_estimado"] = valor_estimado
    out["expected_loss"] = out["prob_perder"] * out["valor_condenacao_estimado"]
    return out


# Funcao pronta para uso em novos dados:
# Ela espera um DataFrame ja com as mesmas colunas cruas de entrada da base.
def prever_custo_esperado(df_entrada):
    return score_expected_loss(df_entrada, RISK_CKPT, COST_CKPT)


# Avaliacao numerica em dados NAO vistos (conjunto de teste).
TEST_SCORED = prever_custo_esperado(DF_TEST)

LIMIAR_ACORDO = 3000
TEST_SCORED["decisao"] = np.where(
    TEST_SCORED["expected_loss"] > LIMIAR_ACORDO,
    "ACORDO",
    "DEFESA",
)

TEST_SCORED["custo_esperado_sem_politica"] = TEST_SCORED["expected_loss"]
TEST_SCORED["custo_esperado_com_politica"] = np.where(
    TEST_SCORED["decisao"] == "ACORDO",
    0.30 * TEST_SCORED["valor_condenacao_estimado"],
    TEST_SCORED["expected_loss"],
)

custo_esperado_sem = TEST_SCORED["custo_esperado_sem_politica"].sum()
custo_esperado_com = TEST_SCORED["custo_esperado_com_politica"].sum()
economia_esperada = custo_esperado_sem - custo_esperado_com

print("===== COMBINACAO DOS MODELOS (TESTE NAO VISTO) =====")
print("Formula: expected_loss = P(perder) x valor_condenacao_estimado")
print("Total de casos teste:", len(TEST_SCORED))
print("Acordos sugeridos:", (TEST_SCORED["decisao"] == "ACORDO").sum())
print("Defesas sugeridas:", (TEST_SCORED["decisao"] == "DEFESA").sum())
print("Custo esperado sem politica:", custo_esperado_sem)
print("Custo esperado com politica:", custo_esperado_com)
print("Economia esperada:", economia_esperada)
print("Economia esperada (%):", (economia_esperada / custo_esperado_sem) * 100 if custo_esperado_sem > 0 else 0)

print("\nAmostra de inferencia combinada:")
print(TEST_SCORED[["prob_perder", "valor_condenacao_estimado", "expected_loss", "decisao"]].head(10))

===== COMBINACAO DOS MODELOS (TESTE NAO VISTO) =====
Formula: expected_loss = P(perder) x valor_condenacao_estimado
Total de casos teste: 12000
Acordos sugeridos: 4893
Defesas sugeridas: 7107
Custo esperado sem politica: 47898057.358297005
Custo esperado com politica: 23977069.284988515
Economia esperada: 23920988.07330849
Economia esperada (%): 49.941457738817554

Amostra de inferencia combinada:
       prob_perder  valor_condenacao_estimado  expected_loss decisao
33176     0.052512               14503.913944     761.623350  DEFESA
11240     0.074155                5525.903839     409.772180  DEFESA
3653      0.029638               10012.382714     296.744070  DEFESA
56554     0.610678                5726.223618    3496.880425  ACORDO
11747     0.648015                8408.977088    5449.146018  ACORDO
10771     0.889155               14393.856171   12798.374700  ACORDO
49594     0.106562               11971.280360    1275.687765  DEFESA
9454      0.011860                8338.361323  